# Predicting default and cancellation (Bondora, 2023–2025)

**Goal:** predict, on the day a loan is issued, (1) whether it will default
within 12 months and (2) whether it will be cancelled.

**Method:** multiple logistic regression (yes/no targets), with multiple
linear regression as a first comparison.

**Data:** `loans_clean.parquet` (410,865 loans, 24 columns).

## 1. Load the data

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_parquet("../data/loans_clean.parquet")
df.shape   # (410865, 24)

(410865, 24)

In [3]:
# checking the Scikit is intalled
import sklearn
print(sklearn.__version__)

1.9.0


## Choosing the inputs

In [ ]:
# Step 3: Only information known on the day the loan is issued (no leakage)
df["log_income"] = np.log1p(df["combined_income"])   # income is log-normal (skew 13.6)
df["log_amount"] = np.log(df["issued_amount"])       # amount is roughly log-normal

num = ["log_income", "log_amount", "initial_interest_rate", "initial_loan_duration"]
cat = ["country", "rating_group"]

df[num + cat].head()

,log_income,log_amount,initial_interest_rate,initial_loan_duration,country,rating_group
1,7.661946,8.329899,0.1927,60,Finland,Unrated
3,7.640373,5.332719,0.1877,60,Finland,D
4,7.535633,6.249975,0.1780,109,Finland,D
5,8.074761,8.329899,0.1927,24,Finland,Unrated
10,7.975451,4.644391,0.1056,60,Netherlands,D


## 3. Choosing the inputs (features)

We only use information that is **known on the day the loan is issued**.
That way the model could also be used for new loan applications.

| Input | Type | Why we use it |
|---|---|---|
| `log_income` | number | Borrower's monthly income (log) |
| `log_amount` | number | Loan amount (log) |
| `initial_interest_rate` | number | Interest rate set by Bondora |
| `initial_loan_duration` | number | Planned length of the loan in months |
| `country` | category | Borrower's country (6 countries) |
| `rating_group` | category | Bondora risk rating AA–HR or Unrated |

**Why log?** Income and loan amount are strongly right-skewed (income skew 13.6),
with a few extremely high values. The log makes the distribution close to normal
so a few very rich borrowers or very large loans do not dominate the model.
Example: `log_income` = 7.66 means an income of about 2,120 EUR per month.

**Columns we do NOT use (leakage):** `loan_status`, `loan_status_risk`,
`months_in_default`, `months_in_default_flag`, `months_on_book`, `principal_debt`,
`principal_balance`, `principal_paid_total`, `interest_paid_total`,
`repaid_amount_total`, `early_repaid_at`.
These are only known **after** the loan has run for a while or after it has defaulted.
Using them would make the model look perfect but it could not work for new loans.

## Tools we used for both the Models

In [ ]:
# Step 4
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

def make_pre():
    # Numbers -> same scale; text -> 0/1 columns (reference: Denmark and rating A)
    return ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat),
    ])

def make_X(data):
    X = data[num + cat].copy()
    for c in cat:
        X[c] = X[c].astype(str)
    return X

## MODEL 1: Predicting default

In [ ]:
# Step 5: Target and Train/Test split
# Only loans with a known 12-month outcome (fair: same observation time for every loan)
data1 = df[df["has_default_within_12_months"].notna()]
X1 = make_X(data1)
y1 = data1["has_default_within_12_months"].astype(int)

X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, stratify=y1, random_state=42)

print(len(data1), "loans | default rate:", round(y1.mean(), 3))

385399 loans | default rate: 0.091


### Result: target and train/test split

- **385,399 loans** have a known 12-month outcome. The other 25,466 loans
  (issued in late 2025) are not yet 12 months old, so we leave them out.
- **Default rate within 12 months: 9.1%.** 34,959 of these loans defaulted within 12 months.
  This is our target (`1` = defaulted within 12 months, `0` = did not).
- We use the **12-month default** instead of `is_default`, so every loan is
  observed for the same time. Otherwise older loans (2023) would look riskier
  just because they had more time to default.

**Train/test split (80% / 20%):**

| Part | Loans | Used for |
|---|---|---|
| Training data | 308,319 | The model learns the patterns here |
| Test data | 77,080 | We check the model on loans it has never seen |

- `stratify=y1` keeps the default rate at **9.1% in both parts**, so the test
  data is a fair copy of the whole dataset.
- `random_state=42` makes the split the same every time we run the notebook,
  so our results can be repeated.

  **AUC- Area under Curve. It tells how well the model ranks risky loans above safe ones: 0.5 is guessing and 1.0 is perfect. Our default model reaches 0.70, so in 70% of cases it ranks a defaulted loan as riskier than a repaid one**

**Note:** only 9.1% of loans default, so the classes are **imbalanced**.
A model that always says "no default" would already be 91% correct, so we
will judge the model with **AUC, recall and precision**, not with accuracy.

## Multiple Linear Regression

In [7]:
# Step 6: Multiple Linear regression (predicting probability of default)
lin1 = Pipeline([("pre", make_pre()), ("model", LinearRegression())]).fit(X1_train, y1_train)
p_lin1 = lin1.predict(X1_test)

print("R2:", round(lin1.score(X1_test, y1_test), 3))
print("Predictions below 0:", round((p_lin1 < 0).mean() * 100, 1), "%")
print("Smallest / largest prediction:", round(p_lin1.min(), 3), "/", round(p_lin1.max(), 3))

R2: 0.054
Predictions below 0: 3.8 %
Smallest / largest prediction: -0.075 / 0.481


### Result: multiple linear regression

- **R² = 0.054:** the model explains only 5.4% of the variation in default,
  because a straight line fits a yes/no target (0 or 1) badly.
- **3.8% of predictions are below 0** (smallest: −0.075). A probability below 0
  is impossible, so the linear model gives results that make no sense.
- **Largest prediction: 0.481.** The model never predicts a default probability
  above 48%, even for the riskiest loans.

**Conclusion:** multiple linear regression is not suitable for a yes/no target.
We therefore use **logistic regression**, which always gives probabilities between 0 and 1.

## 7. Multiple logistic regression (main model)

Logistic regression is made for yes/no targets: it predicts a **probability
of default between 0 and 1** for every loan.

- `class_weight="balanced"`: only 9.1% of loans default. This setting gives the
  defaulted loans more weight, so the model does not just predict "no default"
  for everyone.
- `max_iter=1000`: lets the model take enough steps to find the best coefficients.
- `predict_probability(...)[:, 1]`: the probability of default (class 1) for each test loan.

In [8]:
logit1 = Pipeline([
    ("pre", make_pre()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000)),
]).fit(X1_train, y1_train)

p1 = logit1.predict_proba(X1_test)[:, 1]      # probability of default per loan
print("AUC:", round(roc_auc_score(y1_test, p1), 3))

AUC: 0.704


### Result

**AUC = 0.704.** If we pick one defaulted and one non-defaulted loan at random,
the model ranks the defaulted loan as riskier in **70% of cases**
(0.5 = guessing, 1.0 = perfect). For only 6 inputs, this is a **fair result**,
in the typical range for credit-risk models (0.7–0.8).

Unlike linear regression, all predictions are real probabilities between 0 and 1.

## Evaluation

In [ ]:
# Step 8 :matrix and classification report
y1_pred = (p1 >= 0.5).astype(int)
print(confusion_matrix(y1_test, y1_pred))
print(classification_report(y1_test, y1_pred, digits=3))

[[43024 27064]
 [ 2268  4724]]
              precision    recall  f1-score   support

           0      0.950     0.614     0.746     70088
           1      0.149     0.676     0.244      6992

    accuracy                          0.619     77080
   macro avg      0.549     0.645     0.495     77080
weighted avg      0.877     0.619     0.700     77080



### Result: evaluation (cut-off 0.5)

| | Predicted: no default | Predicted: default |
|---|---|---|
| **Really no default** | 43,024 (correct) | 27,064 (false alarm) |
| **Really default** | 2,268 (missed) | 4,724 (caught) |

- **Recall 0.676:** the model catches **68% of all defaults** (4,724 of 6,992).
- **Precision 0.149:** of the loans it flags, **15% really default**, 1.6× the
  base rate of 9.1%. Loans it calls safe default only 5.0%.
- **Accuracy 0.619** is not a good measure here, because the classes are imbalanced.
  The model is set (`class_weight="balanced"`) to catch defaults, so it accepts
  more false alarms.

**Conclusion:** the model is useful as a **screening tool**. It catches two
thirds of defaults, and flagged loans can get an extra manual check.

**The model catches 68% of all defaults; the loans it flags default 1.6× more often than average, so it works as a screening tool for extra checks**

## Odds ratios: what drives default?

In [ ]:
#Step 9 (a): Odds ratios
names1 = logit1.named_steps["pre"].get_feature_names_out()
odds1 = pd.Series(np.exp(logit1.named_steps["model"].coef_[0]), index=names1).sort_values()
odds1.round(2)

cat__country_Estonia          0.10
cat__country_Netherlands      0.16
cat__country_Finland          0.17
cat__country_Latvia           0.38
cat__rating_group_AA          0.54
num__log_income               0.79
cat__country_Spain            0.84
cat__rating_group_HR          0.96
num__initial_interest_rate    1.14
num__initial_loan_duration    1.22
num__log_amount               1.26
cat__rating_group_B           1.32
cat__rating_group_C           1.79
cat__rating_group_Unrated     1.84
cat__rating_group_D           2.33
cat__rating_group_E           2.85
cat__rating_group_F           3.06
cat__rating_group_G           3.38
dtype: float64

### Result: odds ratios

**How to read:** above 1 = higher risk of default, below 1 = lower risk.
Ratings are compared with **rating A**, countries with **Denmark**.
For the numbers, the effect is per 1 standard deviation (because of `StandardScaler`).
All other inputs are held equal.

- **Rating is the strongest driver.** The odds rise step by step:
  B 1.3× → C 1.8× → D 2.3× → E 2.9× → F 3.1× → **G 3.4×** (vs A).
  AA loans have about half the odds (0.54). Unrated loans: 1.8×.
- **Country matters a lot.** Compared with Denmark: Estonia 0.10, Netherlands 0.16,
  Finland 0.17, Latvia 0.38, so all are much safer than Denmark.
  Spain (0.84) is close to Denmark: its high default rate is mostly explained by
  its unrated and larger loans.
- **Loan terms add some risk:** larger amount 1.26×, longer duration 1.22×,
  higher interest rate 1.14×.
- **Higher income lowers risk** (0.79).
- HR (0.96) is not interpreted: only 19 loans.

**Conclusion:** even when income and loan terms are held equal, **rating and
country remain the strongest drivers of default**. This confirms our SQL and Tableau results.

**Even with all other factors equal, a G-rated loan has 3.4 times the default odds of an A-rated loan, and the country changes the risk by up to 10 times**

**In logistic regression, one category is the reference and the others are compared with it. By default the code picks the first one alphabetically (Denmark, rating A)**.

 **Reference groups:** in logistic regression, one category of each variable is the
reference, and all others are compared with it. The code picks the first one in
alphabetical order, so **Denmark** is the reference country and **rating A** the
reference rating. Changing the reference would only change the comparison point,
not the model or its AUC.

In [ ]:
#step 9 (b): Splitting the test loans into 10 equal groups by predicted risk (1 = riskiest 10%)
risk = pd.DataFrame({"defaulted": y1_test.values, "probability": p1})
risk["risk_group"] = pd.qcut(risk["probability"], 10, labels=range(10, 0, -1))
risk.groupby("risk_group", observed=True)["defaulted"].agg(["count", "mean"]).sort_index()

,count,mean
risk_group,,
10,7708,0.017255
9,7708,0.032304
8,7708,0.043721
7,7708,0.053840
6,7708,0.075117
5,7708,0.084458
4,7708,0.098858
3,7708,0.120005
2,7708,0.140503


### Result: risk groups (deciles)

The 77,080 test loans are split into **10 equal groups of 7,708 loans** by their
predicted risk. Group 1 = the riskiest 10%, group 10 = the safest 10%.
`mean` = the real default rate in each group.

| Risk group | Real default rate |
|---|---|
| **1 (riskiest 10%)** | **24.1%** |
| 2 | 14.1% |
| 3 | 12.0% |
| 4 | 9.9% |
| 5 | 8.4% |
| 6 | 7.5% |
| 7 | 5.4% |
| 8 | 4.4% |
| 9 | 3.2% |
| **10 (safest 10%)** | **1.7%** |

- The default rate **falls steadily from group 1 to group 10**, so the model ranks
  the loans correctly.
- The riskiest 10% default **24.1%**: 2.6× the average (9.1%) and **14× the safest group** (1.7%).
- The riskiest 20% of loans contain **42% of all defaults**.
- The safest 30% of loans contain only **10% of all defaults**.

**Conclusion:** checking only the riskiest 20% of loans would catch almost half of all defaults.

**The riskiest 10% of loans default 24%, fourteen times more than the safest 10% (1.7%), so the model clearly separates risky from safe loans**

In [16]:
#step 9 (c): Precision and recall at different cut-offs
from sklearn.metrics import precision_score, recall_score

for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred = (p1 >= t).astype(int)
    print(f"cut-off {t}: flagged {pred.mean():.1%} | "
          f"recall {recall_score(y1_test, pred):.3f} | precision {precision_score(y1_test, pred):.3f}")

cut-off 0.3: flagged 76.5% | recall 0.931 | precision 0.110
cut-off 0.4: flagged 61.5% | recall 0.847 | precision 0.125
cut-off 0.5: flagged 41.2% | recall 0.676 | precision 0.149
cut-off 0.6: flagged 18.8% | recall 0.404 | precision 0.195
cut-off 0.7: flagged 5.2% | recall 0.168 | precision 0.296


### Result: choosing the cut-off

The model gives each loan a risk score between 0 and 1. The **cut-off** is the score
from which a loan is flagged as risky.

| Cut-off | Loans flagged | Recall (defaults caught) | Precision (flagged loans that default) |
|---|---|---|---|
| 0.3 | 76.5% | 93.1% | 11.0% |
| 0.4 | 61.5% | 84.7% | 12.5% |
| **0.5** | **41.2%** | **67.6%** | **14.9%** |
| 0.6 | 18.8% | 40.4% | 19.5% |
| 0.7 | 5.2% | 16.8% | 29.6% |

- A **lower** cut-off catches more defaults, but flags many more loans (more false alarms):
  at 0.3, 93% of defaults are caught, but 3 out of 4 loans are flagged.
- A **higher** cut-off flags fewer loans, and those loans are much more likely to default:
  at 0.7, only 5% of loans are flagged, but 29.6% of them default (3× the average of 9.1%).
- The best cut-off is a **business decision**: it depends on how many loans the lender
  can review manually. We use **0.5** as a balanced middle (68% of defaults caught).

  **A low cut-off catches almost all defaults but flags most loans; a high cut-off flags only the riskiest 5%, and almost a third of those really default**

In [17]:
# Step 9 (d): Cross-validation of AUC
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = Pipeline([("pre", make_pre()),
                  ("model", LogisticRegression(class_weight="balanced", max_iter=1000))])
scores = cross_val_score(model, X1, y1, cv=cv, scoring="roc_auc")
print("AUC per fold:", scores.round(3), "| mean:", round(scores.mean(), 3), "| std:", round(scores.std(), 3))

AUC per fold: [0.703 0.702 0.703 0.708 0.703] | mean: 0.704 | std: 0.002


### Result: cross-validation

**What we did:** the data is split into **5 equal parts (folds)**. The model is trained
5 times, each time on 4 parts, and tested on the 5th part it has not seen.
So every loan is used for testing exactly once. `StratifiedKFold` keeps the
9.1% default rate the same in every fold.

| Fold | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| AUC | 0.703 | 0.702 | 0.703 | 0.708 | 0.703 |

- **Mean AUC: 0.704**, the same as our single train/test split (0.704).
- **Standard deviation: 0.002.** The AUC hardly changes between the folds
  (0.702 to 0.708).

**Conclusion:** the model is **stable**. The result does not depend on a lucky split,
so we can trust that it would perform about the same on new loans.

**Tested 5 times on different parts of the data, the model always reaches an AUC of about 0.70, so the result is stable and not a lucky spli**

## MODEL 2: Predicting cancellation

In [ ]:
#step 10:Target and Train/Test split
# All loans: the cancellation outcome is known for every loan
data2 = df
X2 = make_X(data2)
y2 = data2["is_cancelled"].astype(int)

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, stratify=y2, random_state=42)

print(len(data2), "loans | cancellation rate:", round(y2.mean(), 3))

410865 loans | cancellation rate: 0.041


## Using Logistic Regression


In [ ]:
#Step 11: Logistic regression (predicting probability of cancellation)
logit2 = Pipeline([
    ("pre", make_pre()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000)),
]).fit(X2_train, y2_train)

p2 = logit2.predict_proba(X2_test)[:, 1]      # probability of cancellation per loan
print("AUC:", round(roc_auc_score(y2_test, p2), 3))

AUC: 0.651


### Result: logistic regression for cancellation

**AUC = 0.651.** If we pick one cancelled and one non-cancelled loan at random,
the model ranks the cancelled loan higher in **65% of cases**
(0.5 = guessing, 1.0 = perfect).

This is **weaker than the default model (AUC 0.70)**. Cancellation is harder to
predict from our 6 inputs, probably because it depends on the borrower's personal
decision after payout (e.g. finding cheaper money elsewhere), which is not in the data.

**The cancellation model reaches an AUC of 0.65, weaker than the default model (0.70), so cancellation is harder to predict from loan and borrower data**

## Evaluation

In [ ]:
#step 12: matrix and classification report
y2_pred = (p2 >= 0.5).astype(int)
print(confusion_matrix(y2_test, y2_pred))
print(classification_report(y2_test, y2_pred, digits=3))

[[49178 29638]
 [ 1355  2002]]
              precision    recall  f1-score   support

           0      0.973     0.624     0.760     78816
           1      0.063     0.596     0.114      3357

    accuracy                          0.623     82173
   macro avg      0.518     0.610     0.437     82173
weighted avg      0.936     0.623     0.734     82173



### Result: evaluation for cancellation (cut-off 0.5)

| | Predicted: not cancelled | Predicted: cancelled |
|---|---|---|
| **Really not cancelled** | 49,178 (correct) | 29,638 (false alarm) |
| **Really cancelled** | 1,355 (missed) | 2,002 (caught) |

- **Recall 0.596:** the model catches **60% of all cancellations** (2,002 of 3,357).
- **Precision 0.063:** of the loans it flags, **6.3% are really cancelled**,
  1.5× the base rate of 4.1%. Loans it calls safe are cancelled only 2.7% of the time.
- **Accuracy 0.623** is not a good measure, because only 4.1% of loans are cancelled.

**Conclusion:** the cancellation model is **weaker than the default model**
(recall 60% vs 68%, precision 6% vs 15%). It gives a rough signal, but it raises
many false alarms, so it is less useful in practice.

**The cancellation model catches 60% of cancellations, but only 6% of the flagged loans are really cancelled, so it’s a weaker signal than the default model**

## Odds ratios: what drives cancellation?

In [14]:
#Step 13: Odds ratios
names2 = logit2.named_steps["pre"].get_feature_names_out()
odds2 = pd.Series(np.exp(logit2.named_steps["model"].coef_[0]), index=names2).sort_values()
odds2.round(2)

num__log_amount               0.72
num__initial_loan_duration    0.77
cat__rating_group_C           0.85
cat__rating_group_D           0.89
cat__rating_group_E           0.89
cat__rating_group_F           0.93
cat__rating_group_Unrated     0.93
num__initial_interest_rate    0.94
cat__rating_group_B           1.01
num__log_income               1.08
cat__rating_group_HR          1.14
cat__rating_group_AA          1.21
cat__rating_group_G           1.25
cat__country_Spain            1.54
cat__country_Netherlands      1.62
cat__country_Finland          2.70
cat__country_Latvia           3.22
cat__country_Estonia          3.50
dtype: float64

### Result: odds ratios for cancellation

**How to read:** above 1 = more likely to be cancelled, below 1 = less likely.
Countries are compared with **Denmark**, ratings with **rating A**.
For the numbers, the effect is per 1 standard deviation. All other inputs are held equal.

- **Country is the strongest driver.** Compared with Denmark, cancellation odds are
  much higher in **Estonia (3.5×)**, **Latvia (3.2×)** and **Finland (2.7×)**,
  and somewhat higher in the Netherlands (1.6×) and Spain (1.5×).
- **Larger and longer loans are cancelled less:** loan amount 0.72,
  loan duration 0.77. Borrowers with big, long loans rarely return the money early.
- **Rating has only a small effect** (between 0.85 and 1.25). Unlike default,
  cancellation does not depend much on the risk rating.
- **Interest rate (0.94) and income (1.08) have almost no effect.**

**Conclusion:** cancellation is driven mainly by **country and loan size**, not by
risk. This is very different from default, where the **rating** is the strongest driver.
So default and cancellation are two different behaviours and need separate models.

**Cancellation depends mainly on the country and the loan size, not on the rating, so it’s a different behaviour from default.**

## Conclusion

- Default and cancellation are yes/no outcomes, so we used **logistic regression**.
  Linear regression explained only 5% (R² 0.054) and gave impossible negative
  probabilities (3.8% below 0).
- **Default model:** AUC 0.70, stable in 5-fold cross-validation (0.704 ± 0.002).
  It catches 68% of defaults at cut-off 0.5. The riskiest 10% of loans default 24.1%,
  the safest 10% only 1.7% (14× difference).
- **Drivers of default:** rating (G: 3.4× the odds of rating A; AA: 0.54×) and country
  (Estonia 0.10, Finland 0.17, Netherlands 0.16 vs Denmark), even with income and
  loan terms held equal.
- **Cancellation model:** weaker (AUC 0.65); driven by country (Estonia 3.5×, Latvia 3.2×,
  Finland 2.7× vs Denmark) and loan size, not by rating.
- **Use:** a screening tool. Flagged loans get a manual check; the cut-off is a
  business choice (how many loans can be reviewed).
- **Limitations:** only 6 inputs; association, not cause; scores are for ranking,
  not exact probabilities; results for Bondora's peer-to-peer market may not
  transfer to banks.

## Part 2: Building the Tableau part
## Step A: Export the model results from the notebook

In [18]:
# Step 14: Export model results for Tableau
X_all = make_X(df)

scores = df[["loan_id", "country", "rating_group", "issued_amount", "combined_income",
             "initial_interest_rate", "initial_loan_duration",
             "has_default_within_12_months", "is_cancelled"]].copy()
scores["issued_year"] = df["loan_issued_at"].dt.year

# 1) Risk score + risk group (1 = riskiest 10%) for EVERY loan
scores["default_score"] = logit1.predict_proba(X_all)[:, 1]
scores["cancel_score"]  = logit2.predict_proba(X_all)[:, 1]
scores["default_risk_group"] = pd.qcut(scores["default_score"], 10, labels=range(10, 0, -1)).astype(int)
scores["cancel_risk_group"]  = pd.qcut(scores["cancel_score"], 10, labels=range(10, 0, -1)).astype(int)

# Which loans were train / test (only test loans are used to judge the model)
scores["default_set"] = np.where(scores.index.isin(X1_test.index), "test",
                        np.where(scores.index.isin(X1_train.index), "train", "no 12m outcome yet"))
scores["cancel_set"] = np.where(scores.index.isin(X2_test.index), "test", "train")
scores.to_csv("../data/model_scores.csv", index=False)

# 2) Odds ratios of both models
def odds_table(model, name):
    o = pd.Series(np.exp(model.named_steps["model"].coef_[0]),
                  index=model.named_steps["pre"].get_feature_names_out())
    t = o.rename("odds_ratio").reset_index().rename(columns={"index": "feature"})
    t["feature"] = t["feature"].str.replace("cat__", "").str.replace("num__", "")
    t["model"] = name
    return t

pd.concat([odds_table(logit1, "Default (12 months)"),
           odds_table(logit2, "Cancellation")]).to_csv("../data/model_odds_ratios.csv", index=False)

# 3) Recall and precision at every cut-off (test data)
rows = []
for name, y_t, p in [("Default (12 months)", y1_test, p1), ("Cancellation", y2_test, p2)]:
    for t in np.arange(0.1, 0.95, 0.05).round(2):
        pred = (p >= t).astype(int)
        rows.append({"model": name, "cutoff": t, "flagged_pct": pred.mean(),
                     "recall": recall_score(y_t, pred),
                     "precision": precision_score(y_t, pred, zero_division=0)})
pd.DataFrame(rows).to_csv("../data/model_cutoffs.csv", index=False)

print(scores.shape, scores["default_set"].value_counts().to_dict())

(410865, 16) {'train': 308319, 'test': 77080, 'no 12m outcome yet': 25466}


### Result: export for Tableau

Three files were saved in `data/`:

| File | Contents |
|---|---|
| `model_scores.csv` | One row per loan (410,865): default and cancellation score, risk group (1 = riskiest 10%), and whether the loan was in the training or test data |
| `model_odds_ratios.csv` | Odds ratio of every input, for both models |
| `model_cutoffs.csv` | Recall and precision at cut-offs from 0.1 to 0.9 (test data) |

- **308,319** loans were used for training and **77,080** for testing the default model.
- **25,466** young loans (late 2025) have no 12-month outcome yet. The model still gives
  them a risk score, so they can be monitored early.
- Only **test** loans are used in Tableau to show how good the model is.